<a href="https://colab.research.google.com/github/walmirpacheco/Assistencia_Virtual/blob/main/Assistencia_Virtual.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# INSTALAÇÕES
import sys
import subprocess
import time
import os
import base64
import io
import tempfile
import webbrowser
from IPython.display import HTML, Audio, display
from google.colab import output

In [2]:
# unção para instalar pacotes
def install_packages():
    packages = [
        'gtts',
        'SpeechRecognition',
        'pyjokes',
        'wikipedia'
    ]

    print("📦 Instalando dependências...")
    for package in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

    # Instalar ffmpeg e portaudio para suporte de áudio
    subprocess.run(['apt-get', 'update', '-qq'], capture_output=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'portaudio19-dev'], capture_output=True)

    print("✅ Dependências instaladas com sucesso!")

In [3]:
# Executar instalaçõe
install_packages()
import speech_recognition as sr
from gtts import gTTS
import pyjokes
import wikipedia
import numpy as np

📦 Instalando dependências...
✅ Dependências instaladas com sucesso!


In [4]:
# CONFIGURAÇÕES
class Config:
    LANGUAGE = 'pt'  # Idioma principal (português)
    SECONDARY_LANG = 'en'  # Idioma secundário (inglês)
    RECORD_SECONDS = 6  # Segundos de gravação
    SAMPLE_RATE = 16000  # Taxa de amostragem

In [5]:
# TEXTO PARA ÁUDIO
class TextToSpeech:
    def __init__(self, lang=Config.LANGUAGE):
        self.lang = lang

    def speak(self, text, show_text=True):
        """Converte texto em áudio e reproduz"""
        try:
            if show_text:
                print(f"\n🔊 Assistente: {text}")

            # Criar arquivo de áudio temporário
            tts = gTTS(text=text, lang=self.lang, slow=False)
            filename = "temp_speech.mp3"
            tts.save(filename)

            # Reproduzir áudio
            display(Audio(filename, autoplay=True))

            # Aguardar término da reprodução
            time.sleep(min(len(text) * 0.08, 5))

            # Limpar arquivo
            os.remove(filename)

        except Exception as e:
            print(f"❌ Erro ao reproduzir áudio: {e}")
            print(f"Texto: {text}")

In [6]:
# COMANDOS E AUTOMAÇÕES
class CommandProcessor:
    def __init__(self, tts):
        self.tts = tts
        self.commands = {
            'youtube': self.open_youtube,
            'pesquisar': self.search_wikipedia,
            'buscar': self.search_wikipedia,
            'search': self.search_wikipedia,
            'piada': self.tell_joke,
            'joke': self.tell_joke,
            'horas': self.get_time,
            'que horas são': self.get_time,
            'sair': self.exit_assistant,
            'tchau': self.exit_assistant,
            'exit': self.exit_assistant,
            'ajuda': self.show_help,
            'help': self.show_help,
            'abrir google': self.open_google,
            'google': self.open_google,
            'abrir github': self.open_github
        }

    def open_youtube(self, *args):
        """Abre o YouTube"""
        self.tts.speak("Abrindo YouTube")
        webbrowser.open("https://www.youtube.com")
        print("🌐 YouTube aberto no navegador")

    def open_google(self, *args):
        """Abre o Google"""
        self.tts.speak("Abrindo Google")
        webbrowser.open("https://www.google.com")
        print("🌐 Google aberto no navegador")

    def open_github(self, *args):
        """Abre o GitHub"""
        self.tts.speak("Abrindo GitHub")
        webbrowser.open("https://github.com")
        print("🌐 GitHub aberto no navegador")

    def search_wikipedia(self, text):
        """Pesquisa na Wikipedia"""
        # Extrair o termo de pesquisa
        query = text.lower()
        for palavra in ['pesquisar', 'buscar', 'search', 'sobre', 'por', 'o que é']:
            query = query.replace(palavra, "")
        query = query.strip()

        if query and len(query) > 2:
            self.tts.speak(f"Procurando na Wikipedia sobre {query}")
            print(f"🔍 Pesquisando: {query}")

            try:
                wikipedia.set_lang('pt')
                result = wikipedia.summary(query, sentences=3)
                print(f"\n📖 Wikipedia:\n{result[:500]}...")
                self.tts.speak(f"De acordo com a Wikipedia: {result[:200]}")
            except wikipedia.exceptions.DisambiguationError as e:
                options = e.options[:5]
                self.tts.speak(f"Encontrei múltiplos resultados. Tente: {', '.join(options)}")
                print(f"Opções disponíveis: {options}")
            except wikipedia.exceptions.PageError:
                self.tts.speak(f"Não encontrei informações sobre {query}")
            except Exception as e:
                print(f"Erro na pesquisa: {e}")
                self.tts.speak("Desculpe, ocorreu um erro na pesquisa")
        else:
            self.tts.speak("Por favor, diga o que deseja pesquisar")

    def tell_joke(self, *args):
        """Conta uma piada"""
        try:
            # Tentar piada em português
            joke = pyjokes.get_joke('pt')
            print(f"\n😂 {joke}")
            self.tts.speak(joke)
        except:
            # Fallback para inglês
            joke = pyjokes.get_joke()
            print(f"\n😂 {joke}")
            self.tts.speak(joke)

    def get_time(self, *args):
        """Informa a hora atual"""
        from datetime import datetime
        now = datetime.now()
        time_str = now.strftime("%H:%M")
        self.tts.speak(f"Agora são {time_str}")
        print(f"🕐 Hora atual: {time_str}")

    def show_help(self, *args):
        """Mostra os comandos disponíveis"""
        help_text = """Comandos disponíveis:
        • youtube - Abre o YouTube
        • pesquisar [assunto] - Pesquisa na Wikipedia
        • piada - Conta uma piada
        • horas - Diz a hora atual
        • abrir google - Abre o Google
        • abrir github - Abre o GitHub
        • ajuda - Mostra esta lista
        • sair - Encerra o assistente"""

        print(f"\n📋 {help_text}")
        self.tts.speak("Comandos disponíveis. Digite ajuda para ver a lista completa")

    def exit_assistant(self, *args):
        """Encerra o assistente"""
        self.tts.speak("Até logo! Tenha um bom dia")
        print("👋 Assistente encerrado. Execute novamente para reativar.")
        return True  # Sinaliza para encerrar

    def process(self, text):
        """Processa o comando recebido"""
        print(f"\n📝 Processando: '{text}'")

        text_lower = text.lower()

        # Verificar cada comando
        for keyword, handler in self.commands.items():
            if keyword in text_lower:
                result = handler(text)
                if result is True:  # Se o comando retornar True, encerra
                    return True
                return False

        # Comando não reconhecido
        self.tts.speak("Desculpe, não entendi esse comando. Diga ajuda para ver os comandos disponíveis")
        return False

In [7]:
# ÁUDIO E RECONHECIMENTO
class AudioProcessor:
    def __init__(self, command_processor):
        self.command_processor = command_processor
        self.recognizer = sr.Recognizer()
        # Ajustar sensibilidade
        self.recognizer.energy_threshold = 300
        self.recognizer.dynamic_energy_threshold = True

    def text_to_speech_direct(self):
        """Permite digitar texto para converter em áudio"""
        print("\n" + "="*50)
        print("📝 MODO TEXTO-TO-SPEECH")
        print("="*50)
        print("Digite algo e o assistente falará")
        print("(Digite 'sair' para voltar ao modo voz)")
        print("="*50)

        while True:
            text = input("\n✏️ Digite seu texto: ").strip()
            if text.lower() in ['sair', 'exit', 'voltar']:
                break
            if text:
                self.command_processor.tts.speak(text)

    def listen_from_microphone(self):
        """Captura áudio do microfone via navegador"""
        # Interface HTML para gravação
        audio_html = """
        <script>
        let mediaRecorder;
        let audioChunks = [];

        async function startRecording() {
            try {
                const stream = await navigator.mediaDevices.getUserMedia({
                    audio: {
                        echoCancellation: true,
                        noiseSuppression: true,
                        sampleRate: 16000
                    }
                });

                mediaRecorder = new MediaRecorder(stream, {
                    mimeType: 'audio/webm'
                });

                audioChunks = [];

                mediaRecorder.addEventListener("dataavailable", event => {
                    if (event.data.size > 0) {
                        audioChunks.push(event.data);
                    }
                });

                mediaRecorder.addEventListener("stop", async () => {
                    const audioBlob = new Blob(audioChunks, { type: 'audio/webm' });
                    const reader = new FileReader();
                    reader.onloadend = () => {
                        const base64Audio = reader.result.split(',')[1];
                        google.colab.kernel.invokeFunction('notebook.process_audio', [base64Audio], {});
                    };
                    reader.readAsDataURL(audioBlob);
                    document.getElementById('status').innerHTML = '🎤 Processando áudio...';
                    document.getElementById('status').style.color = 'blue';
                });

                mediaRecorder.start();
                document.getElementById('status').innerHTML = '🔴 Gravando... Fale agora!';
                document.getElementById('status').style.color = 'red';

                setTimeout(() => {
                    if (mediaRecorder && mediaRecorder.state === 'recording') {
                        mediaRecorder.stop();
                        stream.getTracks().forEach(track => track.stop());
                        document.getElementById('status').innerHTML = '⏸️ Gravação concluída!';
                        document.getElementById('status').style.color = 'green';
                    }
                }, 6000);

            } catch (err) {
                console.error('Erro:', err);
                document.getElementById('status').innerHTML = '❌ Erro: ' + err.message;
                document.getElementById('status').style.color = 'red';
            }
        }
        </script>

        <div style="text-align: center; padding: 20px; border: 2px solid #4CAF50; border-radius: 10px; margin: 10px 0; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);">
            <h2 style="color: white;">🎙️ Assistente Virtual</h2>
            <button onclick="startRecording()" style="padding: 15px 30px; font-size: 18px; background-color: #4CAF50; color: white; border: none; border-radius: 50px; cursor: pointer; margin: 10px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); transition: transform 0.2s;">
                🎤 Clique e fale (6 segundos)
            </button>
            <p id="status" style="margin-top: 10px; font-size: 14px; font-weight: bold; color: white;">✅ Clique no botão e comece a falar</p>
            <p style="color: white; font-size: 12px;">⚡ Fale claramente após o botão ficar vermelho</p>
        </div>
        """

        return audio_html

    def process_audio_callback(self, audio_base64):
        """Callback para processar áudio gravado"""
        try:
            # Decodificar áudio
            audio_bytes = base64.b64decode(audio_base64)

            # Converter para formato WAV usando ffmpeg
            with tempfile.NamedTemporaryFile(suffix='.webm', delete=False) as f:
                f.write(audio_bytes)
                webm_file = f.name

            wav_file = webm_file.replace('.webm', '.wav')

            # Converter com ffmpeg
            import subprocess
            subprocess.run([
                'ffmpeg', '-i', webm_file,
                '-acodec', 'pcm_s16le',
                '-ar', '16000',
                '-ac', '1',
                wav_file,
                '-y',
                '-loglevel', 'quiet'
            ], capture_output=True)

            # Reconhecimento de voz
            with sr.AudioFile(wav_file) as source:
                self.recognizer.adjust_for_ambient_noise(source, duration=0.5)
                audio = self.recognizer.record(source)

                try:
                    # Tentar português
                    text = self.recognizer.recognize_google(audio, language='pt-BR')
                    print(f"\n✅ Reconhecido: {text}")
                    self.command_processor.process(text)

                except sr.UnknownValueError:
                    # Tentar inglês
                    try:
                        text = self.recognizer.recognize_google(audio, language='en-US')
                        print(f"\n✅ Reconhecido: {text}")
                        self.command_processor.process(text)
                    except:
                        print("❌ Não foi possível reconhecer o áudio")
                        self.command_processor.tts.speak("Desculpe, não entendi o que você disse")

                except sr.RequestError as e:
                    print(f"❌ Erro de conexão: {e}")
                    self.command_processor.tts.speak("Erro de conexão com o serviço de reconhecimento")

            # Limpar arquivos temporários
            os.unlink(webm_file)
            if os.path.exists(wav_file):
                os.unlink(wav_file)

        except Exception as e:
            print(f"❌ Erro ao processar áudio: {e}")

In [14]:
# ASSISTENTE PRINCIPAL
class VirtualAssistant:
    def __init__(self):
        print("\n" + "="*60)
        print("🎙️ ASSISTENTE VIRTUAL COMPLETO")
        print("="*60)
        print("✅ Inicializando...")

        self.tts = TextToSpeech()
        self.command_processor = CommandProcessor(self.tts)
        self.audio_processor = AudioProcessor(self.command_processor)

        print("✅ Assistente pronto!")

    def show_welcome(self):
        """Mostra mensagem de boas-vindas"""
        welcome_msg = "Olá! Sou seu assistente virtual. Clique no botão e fale um comando"
        print(f"\n🔊 {welcome_msg}")
        self.tts.speak(welcome_msg, show_text=False)

    def show_menu(self):
        """Mostra o menu de comandos"""
        menu = """
╔══════════════════════════════════════════════╗
║           📋 COMANDOS DISPONÍVEIS            ║
╠══════════════════════════════════════════════╣
║  🎤 YouTube        → Abre o YouTube          ║
║  🔍 Pesquisar X    → Busca na Wikipedia      ║
║  😂 Piada          → Conta uma piada         ║
║  🕐 Horas          → Diz a hora atual        ║
║  🌐 Abrir Google   → Abre o Google           ║
║  💻 Abrir GitHub   → Abre o GitHub           ║
║  ❓ Ajuda          → Mostra este menu        ║
║  👋 Sair           → Encerra o assistente    ║
╠══════════════════════════════════════════════╣
║  📝 Modo texto     → Digite para falar       ║
╚══════════════════════════════════════════════╝
        """
        print(menu)

    def run(self):
        """Executa o assistente principal"""
        self.show_welcome()
        self.show_menu()

        # Registrar callback
        output.register_callback('notebook.process_audio', self.audio_processor.process_audio_callback)

        # Exibir interface
        display(HTML(self.audio_processor.listen_from_microphone()))

        print("\n" + "="*60)
        print("🎯 ASSISTENTE PRONTO!")
        print("="*60)
        print("\n📌 COMO USAR:")
        print("  1️⃣ Clique no botão verde acima")
        print("  2️⃣ Fale seu comando claramente")
        print("  3️⃣ Aguarde o processamento")
        print("  4️⃣ Ouça a resposta do assistente")
        print("\n📝 OU USE TEXTO:")
        print("  Execute: assistant.audio_processor.text_to_speech_direct()")
        print("  Para digitar textos que serão falados")
        print("="*60)
        print("\n✅ Pronto! Clique no botão para começar.\n")

        # Loop para manter o assistente ativo
        while True:
            try:
                time.sleep(1)
            except KeyboardInterrupt:
                print("\n👋 Encerrando assistente...")
                break

In [16]:
# EXECUÇÃO
if __name__ == "__main__":
    assistant = VirtualAssistant()
    assistant.run()


👋 Encerrando assistente...

✅ Reconhecido: YouTube

📝 Processando: 'YouTube'

🔊 Assistente: Abrindo YouTube


🌐 YouTube aberto no navegador

✅ Reconhecido: piada

📝 Processando: 'piada'

😂 Why are you always smiling? That's just my... regular expression.

🔊 Assistente: Why are you always smiling? That's just my... regular expression.



✅ Reconhecido: Abrir Google

📝 Processando: 'Abrir Google'

🔊 Assistente: Abrindo Google


🌐 Google aberto no navegador

✅ Reconhecido: horas

📝 Processando: 'horas'

🔊 Assistente: Agora são 19:01


🕐 Hora atual: 19:01

✅ Reconhecido: sair

📝 Processando: 'sair'

🔊 Assistente: Até logo! Tenha um bom dia


👋 Assistente encerrado. Execute novamente para reativar.


In [18]:
assistant.audio_processor.text_to_speech_direct()


📝 MODO TEXTO-TO-SPEECH
Digite algo e o assistente falará
(Digite 'sair' para voltar ao modo voz)

✏️ Digite seu texto: Bom dia pessoal, bora estudar!

🔊 Assistente: Bom dia pessoal, bora estudar!



✏️ Digite seu texto: exit
